In [19]:
#import y conexion

import requests
from bs4 import BeautifulSoup
import psycopg2
from psycopg2.extras import execute_values
import time

#conexion a posgresql
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="books_pipeline",
    user="postgres",
    password="postgres"
)
cur = conn.cursor()
print("Conexion exitosa a PostgreSQL")

Conexion exitosa a PostgreSQL


In [ ]:
#scraping de categorias

BASE_URL = "https://books.toscrape.com/"

#descargar la pagina principal
response = requests.get(BASE_URL)
soup = BeautifulSoup(response.text, "html.parser")

#acceder a las categorias 
sidebar = soup.find("ul", class_="nav nav-list")
category_links = sidebar.find_all("li")[1:]

#extraer nombre y url relativa de cada categoria
categories = []
for li in category_links:
    a = li.find("a")
    name = a.text.strip()          #strip() elimina espacios y saltos de linnea
    url  = BASE_URL + a["href"]    #construir la URL completa
    categories.append((name, url))

print(f"Categorias encontradas: {len(categories)}")
print(categories[:3])  #  >:3

Categorias encontradas: 50
[('Travel', 'https://books.toscrape.com/catalogue/category/books/travel_2/index.html'), ('Mystery', 'https://books.toscrape.com/catalogue/category/books/mystery_3/index.html'), ('Historical Fiction', 'https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html')]


In [18]:
#insertar categorias

execute_values(
    cur,
    "INSERT INTO categories (name) VALUES %s ON CONFLICT DO NOTHING",
    [(cat[0],) for cat in categories]  #lista de tuplas
)
conn.commit()
print(f"{len(categories)} categorías insertadas")

50 categorías insertadas


In [20]:
#scraping de libros

#mapa para convertir texto a numero segun el rating
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def scrape_books_from_category(category_name, category_url):
    """
    recorre todas las paginas de una categoria y devuelve una lista de
    dicts con los datos de cada libro
    """
    books = []
    url = category_url

    while url:  #mientras hayan paginas
        response = requests.get(url)
        soup = BeautifulSoup(response.text, "html.parser")

        #cada libro es un article class = product_pod
        articles = soup.find_all("article", class_="product_pod")

        for article in articles:
            #extrer titulo de la etiqueta h3 
            title = article.h3.a["title"]

            #extraer precio dentro de los price_color
            price_text = article.find("p", class_="price_color").text
            price = float(price_text.replace("£", "").replace("Â", "").strip())

            #extraer la calificacion de estrellas rating
            rating_class = article.find("p", class_="star-rating")["class"][1]
            rating = RATING_MAP.get(rating_class, 0)

            #url del libro para despues buscar cosas a detalles
            relative_url = article.h3.a["href"].replace("../../../", "catalogue/")
            book_url = BASE_URL + relative_url

            books.append({
                "title": title,
                "price": price,
                "rating": rating,
                "category": category_name,
                "url": book_url
            })

        #buscar el boton next
        next_btn = soup.find("li", class_="next")
        if next_btn:
            next_href = next_btn.find("a")["href"]
            #reemplazar el filename de la url principal
            url = url.rsplit("/", 1)[0] + "/" + next_href
        else:
            url = None  #si no hay paginas off

        time.sleep(0.3)  #minipausa para no saturar el servidor

    return books


#se ejecuta para cada categoria

#primero traer el id de cada categoria desde la base de datod para el FK
cur.execute("SELECT id, name FROM categories")
cat_db = {row[1]: row[0] for row in cur.fetchall()}

all_books = []
for name, url in categories:
    print(f"Scrapeando: {name}...", end=" ")
    books = scrape_books_from_category(name, url)
    print(f"{len(books)} libros")
    all_books.append((name, books))

total = sum(len(b[1]) for b in all_books)
print(f"\nTotal de libros scrapeados: {total}")

Scrapeando: Travel... 11 libros
Scrapeando: Mystery... 32 libros
Scrapeando: Historical Fiction... 26 libros
Scrapeando: Sequential Art... 75 libros
Scrapeando: Classics... 19 libros
Scrapeando: Philosophy... 11 libros
Scrapeando: Romance... 35 libros
Scrapeando: Womens Fiction... 17 libros
Scrapeando: Fiction... 65 libros
Scrapeando: Childrens... 29 libros
Scrapeando: Religion... 7 libros
Scrapeando: Nonfiction... 110 libros
Scrapeando: Music... 13 libros
Scrapeando: Default... 152 libros
Scrapeando: Science Fiction... 16 libros
Scrapeando: Sports and Games... 5 libros
Scrapeando: Add a comment... 67 libros
Scrapeando: Fantasy... 48 libros
Scrapeando: New Adult... 6 libros
Scrapeando: Young Adult... 54 libros
Scrapeando: Science... 14 libros
Scrapeando: Poetry... 19 libros
Scrapeando: Paranormal... 1 libros
Scrapeando: Art... 8 libros
Scrapeando: Psychology... 7 libros
Scrapeando: Autobiography... 9 libros
Scrapeando: Parenting... 1 libros
Scrapeando: Adult Fiction... 1 libros
Scrapea

In [ ]:
#insertar libros en en al base de datos
inserted = 0
for category_name, books in all_books:
    category_id = cat_db[category_name]

    rows = [
        (b["title"], b["price"], b["rating"], category_id, b["url"])
        for b in books
    ]

    execute_values(
        cur,
        """
        INSERT INTO books (title, price, rating, category_id, url)
        VALUES %s
        ON CONFLICT DO NOTHING
        """,
        rows
    )
    inserted += len(rows)

conn.commit()
print(f"{inserted} libros insertados en la DB")

1000 libros insertados en la DB


In [22]:
#extraer el autor de la pagina

def get_author_from_book_page(book_url):
    """
    entra a la pagina individual de un libro y extrae el nombre del autor
    que se encuentra en la en <table class="table table-striped">
    """
    try:
        response = requests.get(book_url, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")

        #en el sitio de prueba no hay autores, pero por buena practica lo busco y luego trabajo con la API
        rows = soup.find("table", class_="table table-striped")
        if rows:
            for tr in rows.find_all("tr"):
                th = tr.find("th")
                td = tr.find("td")
                if th and "author" in th.text.lower():
                    return td.text.strip()

        return None  #lo busco por el titulo en la API si no existe

    except requests.exceptions.RequestException as e:
        print(f"Error al acceder {book_url}: {e}")
        return None


#test rapido :3
cur.execute("SELECT title, url FROM books LIMIT 1")
test = cur.fetchone()
print(f"Libro: {test[0]}")
print(f"Autor encontrado: {get_author_from_book_page(test[1])}")

Libro: It's Only the Himalayas
Autor encontrado: None


In [23]:
#consulta a OPEN LIBRARY API por titulo

import json

OPENLIBRARY_SEARCH = "https://openlibrary.org/search.json"
OPENLIBRARY_AUTHOR = "https://openlibrary.org/authors/{}.json"

#cache en memoria para hacer la misma consulta varias veces
author_cache = {}

def get_author_from_openlibrary(title):
    """
    busca un libro por titulo en OPEN LIBRARY
    retorna un dict con los datos del autor, o null si no lo encuentra :3
    """
    try:
        #buscar el libro por título
        params = {"title": title, "limit": 1, "fields": "title,author_name,author_key"}
        response = requests.get(OPENLIBRARY_SEARCH, params=params, timeout=10)
        response.raise_for_status()  #lanza excepción si status >= 400
        data = response.json()

        if not data["docs"]:
            return {"name": title[:50] + " (unknown)", "birth_year": None,
                    "country": None, "external_api_id": None,
                    "total_known_works": None, "api_source": "open_library"}

        doc = data["docs"][0]
        author_name = doc.get("author_name", [None])[0]
        author_key  = doc.get("author_key",  [None])[0]

        if not author_name or not author_key:
            return {"name": "Unknown", "birth_year": None, "country": None,
                    "external_api_id": None, "total_known_works": None,
                    "api_source": "open_library"}

        #si ya se consulto antes al autor usamos el cache
        if author_key in author_cache:
            return author_cache[author_key]

        #consultar detalles del autor
        author_url = OPENLIBRARY_AUTHOR.format(author_key.replace("/authors/", ""))
        a_response = requests.get(author_url, timeout=10)
        a_response.raise_for_status()
        a_data = a_response.json()

        #extraer datos, en caso de no existir esta el get
        birth_year = None
        birth_raw = a_data.get("birth_date", "")
        if birth_raw:
            #birth date, puede venir de diferentes formas como solo el año o mas detalles
            digits = [s for s in birth_raw.split() if s.isdigit() and len(s) == 4]
            birth_year = int(digits[0]) if digits else None

        country = None
        if "location" in a_data:
            country = a_data["location"]

        #consultar cuantas obras conocidas tiene
        works_url = f"https://openlibrary.org/authors/{author_key.replace('/authors/', '')}/works.json?limit=0"
        w_response = requests.get(works_url, timeout=10)
        total_works = None
        if w_response.status_code == 200:
            total_works = w_response.json().get("size", None)

        result = {
            "name":              author_name,
            "birth_year":        birth_year,
            "country":           country,
            "external_api_id":   author_key.replace("/authors/", ""),
            "total_known_works": total_works,
            "api_source":        "open_library"
        }

        author_cache[author_key] = result  #guardar en cache
        time.sleep(0.5)  #respetar el rate limit de OPEN LIBRARY
        return result

    except requests.exceptions.Timeout:
        print(f"Timeout buscando: {title}")
        return {"name": "Unknown", "birth_year": None, "country": None,
                "external_api_id": None, "total_known_works": None,
                "api_source": "open_library_error"}

    except requests.exceptions.RequestException as e:
        print(f"  ⚠️ Error API: {e}")
        return {"name": "Unknown", "birth_year": None, "country": None,
                "external_api_id": None, "total_known_works": None,
                "api_source": "open_library_error"}


#test con un libro conocido
test_author = get_author_from_openlibrary("A Light in the Attic")
print(json.dumps(test_author, indent=2))

{
  "name": "Shel Silverstein",
  "birth_year": null,
  "country": null,
  "external_api_id": "OL548174A",
  "total_known_works": 152,
  "api_source": "open_library"
}


In [24]:
#pipeline completo autores a database

#traer todos los libros con sus id y titulo
cur.execute("SELECT id, title FROM books")
all_books_db = cur.fetchall()

not_found_count = 0
total_books = len(all_books_db)

for i, (book_id, title) in enumerate(all_books_db):

    if i % 50 == 0:
        print(f"  Procesando {i}/{total_books}...")

    #consultar API
    author_data = get_author_from_openlibrary(title)

    if author_data["external_api_id"] is None:
        not_found_count += 1

    #insertar autor si no existe
    cur.execute("""
        INSERT INTO authors (name, birth_year, country, external_api_id,
                             total_known_works, api_source)
        VALUES (%s, %s, %s, %s, %s, %s)
        ON CONFLICT (external_api_id) DO NOTHING
        RETURNING id
    """, (
        author_data["name"],
        author_data["birth_year"],
        author_data["country"],
        author_data["external_api_id"],
        author_data["total_known_works"],
        author_data["api_source"]
    ))

    result = cur.fetchone()

    #si el autor ya existe se busca en external_api_id
    if result:
        author_id = result[0]
    else:
        cur.execute("SELECT id FROM authors WHERE external_api_id = %s",
                    (author_data["external_api_id"],))
        row = cur.fetchone()
        author_id = row[0] if row else None

    #autores sin external_api, los unknows
    #van sin unique contraint, buscando por nombre
    if author_id is None:
        cur.execute("SELECT id FROM authors WHERE name = %s", (author_data["name"],))
        row = cur.fetchone()
        if row:
            author_id = row[0]
        else:
            cur.execute("""
                INSERT INTO authors (name, birth_year, country, external_api_id,
                                     total_known_works, api_source)
                VALUES (%s, %s, %s, %s, %s, %s) RETURNING id
            """, (author_data["name"], author_data["birth_year"], author_data["country"],
                  None, author_data["total_known_works"], author_data["api_source"]))
            author_id = cur.fetchone()[0]

    #insertar en relacion a book_author
    if author_id:
        cur.execute("""
            INSERT INTO book_author (book_id, author_id)
            VALUES (%s, %s)
            ON CONFLICT DO NOTHING
        """, (book_id, author_id))

#commit unico al final
conn.commit()

print(f"\nPipeline completo")
print(f"Autores no encontrados en API: {not_found_count}/{total_books} "
      f"({not_found_count/total_books*100:.1f}%)")

  Procesando 0/1000...
  Procesando 50/1000...
  Procesando 100/1000...
Timeout buscando: The Wicked + The Divine, Vol. 1: The Faust Act (The Wicked + The Divine)
  Procesando 150/1000...
  Procesando 200/1000...
  Procesando 250/1000...
  Procesando 300/1000...
  Procesando 350/1000...
  Procesando 400/1000...
  Procesando 450/1000...
  Procesando 500/1000...
  Procesando 550/1000...
  Procesando 600/1000...
  Procesando 650/1000...
Timeout buscando: The God Delusion
  Procesando 700/1000...
  Procesando 750/1000...
  Procesando 800/1000...
  Procesando 850/1000...
  Procesando 900/1000...
Timeout buscando: Redeeming Love
  Procesando 950/1000...

Pipeline completo
Autores no encontrados en API: 548/1000 (54.8%)
